# Part 4c-exact — Cournot as a true MIQP

### Solving the quadratic directly, and measuring what the approximation cost

Part 4c piecewise-linearised revenue because Gurobi's size-limited `pip` licence rejects quadratic
objectives beyond roughly 150 variables. With a full WLS licence that restriction disappears and
the natural formulation becomes available:

$$\max_{s} \;\; \sum_{rt,p} \omega_p \Big[\big(A_{rt,p} - B_{rt,p}(s_{rt,p} + \bar{q}_{rt,p})\big)\, s_{rt,p}\Big] \;-\; \text{Cost}_r$$

Revenue is **concave** in own quantity, so maximising it is a **convex MIQP** — Gurobi handles it
natively via branch-and-bound with QP relaxations at each node.

This notebook does three things:

1. States the exact MIQP formulation
2. **Validates the piecewise approximation against it** at a scale where both fit the restricted
   licence — which turns out to hold a surprise about approximation error inside a game
3. Runs the full-scale Cournot game as an exact MIQP

## 1. Licence

The exact MIQP needs a full licence. Everything else in the series runs on the restricted `pip`
licence.

In [ ]:
!pip install gurobipy --quiet
import os, math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

params = {
    "WLSACCESSID": "REDACTED-CREDENTIAL-ROTATED",
    "WLSSECRET": "REDACTED-CREDENTIAL-ROTATED",
    "LICENSEID": REDACTED-CREDENTIAL-ROTATED,
}
ENV = gp.Env(params=params)
print("gurobipy", gp.gurobi.version(), "- WLS environment ready")

Two practical notes. Credentials pasted into a notebook travel with the file, so it is worth
rotating this key when the project is shared widely and reading it from a Colab secret or an
environment variable instead of a literal. And every model below is constructed with
`gp.Model(env=ENV)` — a plain `gp.Model()` silently falls back to the restricted licence and will
fail on the quadratic objective.

## 2. The model core

Identical to Parts 4a–4c: the same `add_region` chain, the asymmetric instance (R1 incumbent with
2,600 units of accumulated production, R2 cheaper entrant with 500), both learning channels,
variable-length periods.

`SMALL = True` shrinks the horizon to 3 periods so that the **exact MIQP fits inside the restricted
licence** — that is the configuration used to validate the approximation in §4. Leave it `False`
for the full model.

In [ ]:
SMALL = False          # True -> 3-period horizon, small enough for a restricted licence
if SMALL:
    os.environ['P4_SMALL'] = '1'

In [ ]:
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']

# ---------------- TIME ----------------
import os
# P4_SMALL shrinks the horizon so the EXACT MIQP fits the size-limited licence,
# letting us validate the piecewise-linear revenue against the true quadratic.
BLOCKS = ([(2, 1), (1, 3)] if os.environ.get('P4_SMALL') else [(6, 1), (4, 3), (2, 5), (1, 9)])
LEN, START = [], []
_y = 1
for _c, _L in BLOCKS:
    for _ in range(_c):
        LEN.append(_L); START.append(_y); _y += _L
P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
DR = 0.05
OMEGA = {p: sum(1/(1+DR)**t for t in YEARS[p]) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}
REPORT_UNTIL = 28

# ---------------- TECH ----------------
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0
CRF = DR*(1+DR)**LIFE/((1+DR)**LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF*sum(1/(1+DR)**t for t in range(ONLINE[s, v], ONLINE[s, v]+LIFE)
                      if t <= HORIZON) for s in STAGES for v in P}

In [ ]:
# ---------------- ASYMMETRIC INSTANCE ----------------
# R1 = incumbent upstream processor with accumulated experience.
# R2 = entrant, cheaper to build, trying to move downstream.
FIXED = {('MINE','R1'):900.,('PROC','R1'):1500.,('MFG','R1'):1300.,
         ('MINE','R2'):820.,('PROC','R2'):1350.,('MFG','R2'):1180.}
UNIT  = {('MINE','R1'):7.0,('PROC','R1'):11.0,('MFG','R1'):9.5,
         ('MINE','R2'):6.4,('PROC','R2'):10.0,('MFG','R2'):8.7}
OPEX  = {('MINE','R1'):1.2,('PROC','R1'):2.0,('MFG','R1'):2.4,
         ('MINE','R2'):1.35,('PROC','R2'):2.2,('MFG','R2'):2.6}

LEGACY_CAP = {('MINE','R1'):230,('PROC','R1'):205,('MFG','R1'):155,
              ('MINE','R2'):165,('PROC','R2'):125,('MFG','R2'):100}
LEGACY_RET = {('MINE','R1'):11,('PROC','R1'):14,('MFG','R1'):18,
              ('MINE','R2'):9, ('PROC','R2'):16,('MFG','R2'):22}
LEGACY_BYR = -8
# incumbent starts with accumulated production experience
EXPERIENCE0 = {'R1': 2600.0, 'R2': 500.0}

In [ ]:
# ---------------- EFFICIENCY (yield) ----------------
ETA_CEIL = {'MINE':0.92,'PROC':0.95,'MFG':0.93}
ETA_BASE = {'MINE':0.86,'PROC':0.80,'MFG':0.78}
ALPHA    = {'MINE':0.0,'PROC':0.030,'MFG':0.025}
BETA     = {'MINE':0.0,'PROC':0.010,'MFG':0.008}
DELTA_BAR= {'MINE':0.02,'PROC':0.05,'MFG':0.05}
ETA_FLOOR= 0.60
VINTAGES = [-1] + P
BYEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}
ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s]-ETA_BASE[s])*(1-ALPHA[s])**(BYEAR[v]-1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p]-BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s]-fr)*(1-BETA[s])**age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr+DELTA_BAR[s], aged))

In [ ]:
# ---------------- DEMAND & MARKET ----------------
DEMAND = {}
for r, base, g in [('R1', 100.0, 0.008), ('R2', 75.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base*(1+g)**(t-1) for t in YEARS[p])/LEN[p]
TRANSPORT = {(rf, rt): (0.5 if rf == rt else 2.4) for rf in REGIONS for rt in REGIONS}
PRICE_FIXED = 12.0
PEN_SHORT, PEN_DISPOSE = 90.0, 12.0

In [ ]:
# ---------------- LEARNING ----------------
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX, Q_START, Q_ADD, CAPEX_FLOOR, NBP = 0.15, 300.0, 700.0, 0.60, 9
_bc = -math.log2(1-LR_CAPEX)
K = list(range(NBP))
QBP = [Q_START + Q_ADD*k/(NBP-1) for k in K]

def _cap_unit_mult(q):
    return max(CAPEX_FLOOR, (q/Q_START)**(-_bc))

def _cap_cum_mult(q, n=400):
    if q <= Q_START:
        return 0.0
    h = (q-Q_START)/n
    return sum(0.5*(_cap_unit_mult(Q_START+i*h)+_cap_unit_mult(Q_START+(i+1)*h))*h
               for i in range(n))
CBP = [_cap_cum_mult(q) for q in QBP]

LR_OPEX, OPEX_FLOOR, LAG_YEARS, N_TIERS = 0.18, 0.65, 3, 3
TIER_Q, TIER_M = {}, {}

def set_tiers(top_by_region):
    for r in REGIONS:
        top = max(top_by_region[r], 1.0)
        q1 = top/8.0
        TIER_Q[r] = [q1*2**j for j in range(N_TIERS-1)]
        TIER_M[r] = [max(OPEX_FLOOR, (1-LR_OPEX)**j) for j in range(N_TIERS)]

ACTIVE = {r: [(s, v, p) for s in STAGES for v in VINTAGES for p in P
              if (v == -1 and START[p] <= LEGACY_RET[s, r])
              or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v]+LIFE-1)]
          for r in REGIONS}
VIN = {(r, s, p): [v for (ss, v, pp) in ACTIVE[r] if (ss, pp) == (s, p)]
       for r in REGIONS for s in STAGES for p in P}
BUILD = {r: [(s, v) for s in STAGES for v in P if ONLINE[s, v] <= HORIZON]
         for r in REGIONS}

In [ ]:
def add_region(m, r, learning='both'):
    """Attach one region's vertically-integrated chain to model m. Returns handles."""
    b = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')
    c = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')
    x = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')
    f_mp = m.addVars(P, lb=0.0, name=f'fmp_{r}')
    f_pf = m.addVars(P, lb=0.0, name=f'fpf_{r}')
    sale = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')
    disp = m.addVars(P, lb=0.0, name=f'disp_{r}')

    m.addConstrs((c[s, v] <= CAP_MAX*b[s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c[s, v] >= CAP_MIN*b[s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x[s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c[s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')
    m.addConstrs((gp.quicksum(ETA['MINE', v, p]*x['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == f_mp[p] for p in P),
                 name=f'mine_{r}')
    m.addConstrs((f_mp[p] == gp.quicksum(x['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p]*x['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == f_pf[p] for p in P),
                 name=f'pout_{r}')
    m.addConstrs((f_pf[p] == gp.quicksum(x['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p]*x['MFG', v, p]
                              for v in VIN[r, 'MFG', p])
                  == sale.sum('*', p) + disp[p] for p in P), name=f'mout_{r}')

    # cumulative production (undiscounted), regional scope, with initial experience
    cum = m.addVars(P, lb=0.0, ub=3*CAP_MAX*HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum[p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q]*x['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

    capex = gp.quicksum(MU[s, v]*FIXED[s, r]*b[s, v] for (s, v) in BUILD[r]) \
          + gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                        for (s, v) in BUILD[r] if s not in LEARN_STAGES)
    if learning in ('capacity', 'both'):
        Q = m.addVars(P, lb=Q_START, ub=Q_START+Q_ADD, name=f'Q_{r}')
        Cc = m.addVars(P, lb=0.0, name=f'C_{r}')
        lam = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
        m.addConstrs((lam.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
        m.addConstrs((Q[p] == gp.quicksum(QBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sQ_{r}')
        m.addConstrs((Cc[p] == gp.quicksum(CBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sC_{r}')
        m.addConstrs((Q[p] == Q_START + gp.quicksum(c[s, v] for (s, v) in BUILD[r]
                                                    if s in LEARN_STAGES and v <= p)
                      for p in P), name=f'cc_{r}')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])
        rate = sum(UNIT[s, r] for s in LEARN_STAGES)/len(LEARN_STAGES)
        capex += gp.quicksum(MU['PROC', p]*rate*(Cc[p]-(Cc[p-1] if p > 0 else 0.0))
                             for p in P)
    else:
        capex += gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                             for (s, v) in BUILD[r] if s in LEARN_STAGES)

    if learning in ('production', 'both') and TIER_Q:
        J = list(range(N_TIERS))
        z = m.addVars(P, J, vtype=GRB.BINARY, name=f'z_{r}')
        m.addConstrs((z.sum(p, '*') == 1 for p in P), name=f'ot_{r}')
        LAGP = {p: YEAR_TO_P[max(1, START[p]-LAG_YEARS)] for p in P}
        BIGQ = 3*CAP_MAX*HORIZON + EXPERIENCE0[r]
        m.addConstrs((cum[LAGP[p]] >= TIER_Q[r][j-1] - BIGQ*(1-z[p, j])
                      for p in P for j in J if j > 0), name=f'tf_{r}')
        m.addConstrs((cum[LAGP[p]] <= TIER_Q[r][j] + BIGQ*(1-z[p, j])
                      for p in P for j in J if j < N_TIERS-1), name=f'tc_{r}')
        ts = m.addVars(STAGES, P, J, lb=0.0, name=f'ts_{r}')
        m.addConstrs((ts.sum(s, p, '*') == gp.quicksum(x[s, v, p] for v in VIN[r, s, p])
                      for s in STAGES for p in P), name=f'tss_{r}')
        m.addConstrs((ts[s, p, j] <= 3*CAP_MAX*z[p, j]
                      for s in STAGES for p in P for j in J), name=f'tl_{r}')
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*TIER_M[r][j]*ts[s, p, j]
                           for s in STAGES for p in P for j in J)
    else:
        z = None
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*x[s, v, p] for (s, v, p) in ACTIVE[r])

    trans = gp.quicksum(OMEGA[p]*TRANSPORT[r, rt]*sale[rt, p] for rt in REGIONS for p in P)
    dcost = gp.quicksum(OMEGA[p]*PEN_DISPOSE*disp[p] for p in P)
    revenue = gp.quicksum(OMEGA[p]*PRICE_FIXED*sale[rt, p] for rt in REGIONS for p in P)
    return dict(b=b, c=c, x=x, sale=sale, disp=disp, cum=cum, z=z,
                capex=capex, opex=opex, trans=trans, dcost=dcost, revenue=revenue,
                cost=capex+opex+trans+dcost)

In [ ]:
def solve_planner(w1=0.5, learning='both', mipgap=0.005, quiet=True):
    m = gp.Model(); m.Params.OutputFlag = 0 if quiet else 1; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    short = m.addVars(REGIONS, P, lb=0.0, name='short')
    m.addConstrs((gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS) + short[rt, p]
                  >= DEMAND[rt, p] for rt in REGIONS for p in P), name='demand')
    pen = gp.quicksum(OMEGA[p]*PEN_SHORT*short[rt, p] for rt in REGIONS for p in P)
    m.setObjective(w1*H['R1']['cost'] + (1-w1)*H['R2']['cost'] + pen, GRB.MINIMIZE)
    m.optimize()
    m._H, m._short, m._pen = H, short, pen
    return m

In [ ]:
# ================= 4c: Cournot with endogenous price =================
CHOKE    = 30.0     # price at zero quantity
P_ANCHOR = 13.0     # price when quantity equals the Part 4b demand reference
A_INT = {(rt, p): CHOKE for rt in REGIONS for p in P}
B_SLP = {(rt, p): (CHOKE - P_ANCHOR) / DEMAND[rt, p] for rt in REGIONS for p in P}


NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
def best_response_cournot(r, rival_sales, learning='both', mipgap=0.005):
    """Firm r maximises profit facing linear inverse demand
       p[rt,p] = A - B*(own + rival).
    Revenue is piecewise-linearised in own quantity, keeping the model a MILP."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    h = add_region(m, r, learning)
    s = h['sale']
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            q_bar = rival_sales.get((rt, p), 0.0)
            a_eff = A_INT[rt, p] - B_SLP[rt, p] * q_bar
            smax = max(1e-6, A_INT[rt, p] / B_SLP[rt, p] - q_bar)
            S, R = _rev_breakpoints(a_eff, B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1, name=f'rcvx_{rt}_{p}')
            m.addConstr(s[rt, p] == gp.quicksum(S[k] * mu[rt, p, k] for k in KR),
                        name=f'rS_{rt}_{p}')
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR),
                        name=f'rR_{rt}_{p}')
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h, revenue
    return m

In [ ]:
def cournot_iterate(learning='both', first='R1', max_iter=16, tol=0.5, mipgap=1e-3):
    """Iterated best response under Cournot competition.

    Convergence for a game with CONTINUOUS strategies must be tested with a
    TOLERANCE, not by exact state matching: each best response is a MILP solved to
    a finite gap, so the returned quantities wobble slightly between iterations.
    Exact hashing reads that wobble as a cycle."""
    def dist(a, b):
        return max(abs(a[r][k] - b[r][k]) for r in REGIONS for k in a[r])

    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    plans, hist, log = {}, [], []
    order = [first, 'R2' if first == 'R1' else 'R1']
    for it in range(max_iter):
        prev = {r: dict(sales[r]) for r in REGIONS}
        for r in order:
            other = 'R2' if r == 'R1' else 'R1'
            m = best_response_cournot(r, sales[other], learning=learning, mipgap=mipgap)
            if m.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = {(rt, p): m._h['sale'][rt, p].X for rt in REGIONS for p in P}
            plans[r] = tuple(sorted((s_, v) for (s_, v) in m._h['b']
                                    if m._h['b'][s_, v].X > 0.5))
            log.append(dict(iter=it, firm=r, profit=m.ObjVal,
                            revenue=m._rev.getValue(), cost=m._h['cost'].getValue(),
                            builds=len(plans[r]), sales=sum(sales[r].values()),
                            disposal=sum(m._h['disp'][p].X for p in P)))
        cur = {r: dict(sales[r]) for r in REGIONS}
        if it > 0 and dist(cur, prev) < tol:
            return dict(status='CONVERGED', cycle_len=1, iters=it + 1, log=log,
                        plans=plans, sales=sales, drift=dist(cur, prev))
        for k, past in enumerate(hist):                      # genuine k-cycle, k >= 2
            if dist(cur, past) < tol:
                return dict(status='CYCLE', cycle_len=len(hist) - k, iters=it + 1,
                            log=log, plans=plans, sales=sales)
        hist.append(cur)
    return dict(status='MAX_ITER', iters=max_iter, log=log, plans=plans, sales=sales)

In [ ]:
def market_outcome(sales):
    rows = []
    for rt in REGIONS:
        for p in P:
            q = sum(sales[r][rt, p] for r in REGIONS)
            price = A_INT[rt, p] - B_SLP[rt, p] * q
            rows.append(dict(market=rt, period=p, year=START[p], quantity=q, price=price,
                             consumer_surplus=0.5 * B_SLP[rt, p] * q * q,
                             share_R1=(sales['R1'][rt, p] / q if q > 1e-6 else None)))
    return rows

In [ ]:
def joint_profit_max(learning='both', mipgap=0.005):
    """Collusive benchmark: one decision maker maximising the SUM of both profits."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            smax = A_INT[rt, p] / B_SLP[rt, p]
            S, R = _rev_breakpoints(A_INT[rt, p], B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1)
            m.addConstr(gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS)
                        == gp.quicksum(S[k] * mu[rt, p, k] for k in KR))
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR))
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - gp.quicksum(H[r]['cost'] for r in REGIONS), GRB.MAXIMIZE)
    m.optimize()
    m._H, m._rev = H, revenue
    return m

## 3. The exact MIQP best response

The difference from the piecewise version is four lines. Revenue is written as a `QuadExpr`
directly; there is no `mu` interpolation variable, no convexity constraint, no breakpoint mesh.

`m.addConstrs(... <= choke quantity ...)` remains, keeping price non-negative — without it the
model could in principle sell into a negative price, and while it would never want to, the bound
keeps the relaxation tighter.

Note `env=ENV` threaded through. Every quadratic model must be built in the licensed environment.

In [ ]:
def best_response_cournot_miqp(r, rival_sales, learning='both', mipgap=0.005, env=None):
    """EXACT Cournot best response: revenue kept as a true quadratic.

    Identical to best_response_cournot except that revenue is written directly as
    (A - B*(s + q_bar)) * s instead of being interpolated. Requires a licence that
    permits quadratic objectives at this size (the pip restricted licence does not)."""
    m = gp.Model(env=env) if env is not None else gp.Model()
    m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    h = add_region(m, r, learning)
    s = h['sale']
    m.addConstrs((s[rt, p] <= max(0.0, A_INT[rt, p]/B_SLP[rt, p]
                                  - rival_sales.get((rt, p), 0.0))
                  for rt in REGIONS for p in P), name='choke')
    revenue = gp.QuadExpr()
    for rt in REGIONS:
        for p in P:
            q_bar = rival_sales.get((rt, p), 0.0)
            revenue += OMEGA[p]*((A_INT[rt, p] - B_SLP[rt, p]*q_bar)*s[rt, p]
                                 - B_SLP[rt, p]*s[rt, p]*s[rt, p])
    m.setObjective(revenue - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h, revenue
    return m

In [ ]:
def cournot_iterate_miqp(learning='both', first='R1', max_iter=16, tol=0.5,
                         mipgap=1e-3, env=None):
    def dist(a, b):
        return max(abs(a[r][k] - b[r][k]) for r in REGIONS for k in a[r])
    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    plans, hist, log = {}, [], []
    order = [first, 'R2' if first == 'R1' else 'R1']
    for it in range(max_iter):
        prev = {r: dict(sales[r]) for r in REGIONS}
        for r in order:
            other = 'R2' if r == 'R1' else 'R1'
            m = best_response_cournot_miqp(r, sales[other], learning=learning,
                                           mipgap=mipgap, env=env)
            if m.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = {(rt, p): m._h['sale'][rt, p].X for rt in REGIONS for p in P}
            plans[r] = tuple(sorted((s_, v) for (s_, v) in m._h['b']
                                    if m._h['b'][s_, v].X > 0.5))
            log.append(dict(iter=it, firm=r, profit=m.ObjVal,
                            revenue=m._rev.getValue(), cost=m._h['cost'].getValue(),
                            builds=len(plans[r]), sales=sum(sales[r].values()),
                            disposal=sum(m._h['disp'][p].X for p in P)))
        cur = {r: dict(sales[r]) for r in REGIONS}
        if it > 0 and dist(cur, prev) < tol:
            return dict(status='CONVERGED', cycle_len=1, iters=it+1, log=log,
                        plans=plans, sales=sales, drift=dist(cur, prev))
        for k, past in enumerate(hist):
            if dist(cur, past) < tol:
                return dict(status='CYCLE', cycle_len=len(hist)-k, iters=it+1,
                            log=log, plans=plans, sales=sales)
        hist.append(cur)
    return dict(status='MAX_ITER', iters=max_iter, log=log, plans=plans, sales=sales)

## 4. Validating the approximation

Run both formulations on the **same** instance and compare. This section uses the small
configuration so it reproduces on any licence.

Theory says the piecewise version should **understate** profit: revenue is concave, chords lie below
the curve, and we are maximising. The approximation is therefore *conservative* — never optimistic.

In [ ]:
if not SMALL:
    print("NOTE: set SMALL = True and restart to reproduce this section on a restricted licence.\n"
          "      With a full licence it runs at full scale, but the numbers below are from SMALL.")

zero = {(rt, p): 0.0 for rt in REGIONS for p in P}
exact = best_response_cournot_miqp('R1', zero, learning='none', env=ENV)
print(f"exact MIQP profit {exact.ObjVal:12.4f}   "
      f"sales {sum(exact._h['sale'][rt,p].X for rt in REGIONS for p in P):9.4f}")

In [ ]:
rows = []
for n in [3, 5, 7, 11, 21, 41]:
    NBP_REV = n
    globals()['NBP_REV'] = n
    pw = best_response_cournot('R1', zero, learning='none')
    rows.append(dict(breakpoints=n, pwl_profit=round(pw.ObjVal, 3),
                     error=round(pw.ObjVal - exact.ObjVal, 3),
                     error_pct=round(100*(pw.ObjVal-exact.ObjVal)/exact.ObjVal, 4),
                     sales=round(sum(pw._h['sale'][rt, p].X
                                     for rt in REGIONS for p in P), 3)))
globals()['NBP_REV'] = 7
pd.DataFrame(rows)

**Every error is negative**, confirming the theory: the piecewise revenue never overstates. The
approximation is a valid lower bound on achievable profit.

The magnitude falls sharply with mesh density — about −43% at 3 breakpoints, −0.25% at 7, and
−0.08% at 21 and beyond. The convergence is **not monotone** (7 happens to beat 11), because what
matters is not the number of breakpoints but whether one lands near the optimal quantity. This is
the same lesson as the SOS2 re-meshing in Part 3: placement beats density.

Seven breakpoints — the Part 4c default — costs about a quarter of a percent on a single best
response. That is defensible for the qualitative conclusions drawn there.

### But the error behaves differently inside a game

In [ ]:
a = cournot_iterate_miqp(learning='none', first='R1', max_iter=12, env=ENV)
b = cournot_iterate(learning='none', first='R1', max_iter=12)
la = {L['firm']: L for L in a['log'][-2:]}
lb = {L['firm']: L for L in b['log'][-2:]}
pd.DataFrame([
    dict(method='exact MIQP', status=a['status'],
         profit_R1=round(la['R1']['profit'], 3), profit_R2=round(la['R2']['profit'], 3),
         sales_R1=round(la['R1']['sales'], 3), sales_R2=round(la['R2']['sales'], 3)),
    dict(method='piecewise linear', status=b['status'],
         profit_R1=round(lb['R1']['profit'], 3), profit_R2=round(lb['R2']['profit'], 3),
         sales_R1=round(lb['R1']['sales'], 3), sales_R2=round(lb['R2']['sales'], 3)),
])

**This is the finding worth taking away.** In a single optimisation the piecewise error is signed,
bounded and small. In an **equilibrium** it is neither.

Both methods converge, but to *different* equilibria, and the piecewise version reports profits
**above** the exact ones — the opposite sign to the single-solve error, and by a much larger margin
for R2 (roughly +12%) than R1 (roughly +2%).

The mechanism: each firm's approximated best response is slightly off, which perturbs the *rival's*
problem, which perturbs the response to that, and so on around the loop. The fixed point of a
sequence of slightly-wrong maps is not close to the fixed point of the correct maps in any way the
single-solve error bound controls. Approximation error **propagates through the equilibrium
computation** rather than staying local.

Practical consequence: a discretisation accuracy that is perfectly adequate for one optimisation can
be inadequate for a game built out of the same optimisation. If you must approximate inside a
best-response loop, validate at the **equilibrium** level, not the subproblem level — and if the
qualitative conclusions (who leads, whether output rises with learning) flip between meshes, they
are not conclusions.

## 5. The full-scale game as an exact MIQP

With `SMALL = False` and the WLS environment this runs the complete 13-period model with both
learning channels and no revenue approximation.

In [ ]:
m0 = solve_planner(0.5, learning='capacity')
top = {r: m0._H[r]['cum'][P[-1]].X for r in REGIONS}
set_tiers(top)
print("tier thresholds:", {r: [round(q, 1) for q in TIER_Q[r]] for r in REGIONS})

In [ ]:
rows = []
for first in ['R1', 'R2']:
    res = cournot_iterate_miqp(first=first, max_iter=16, env=ENV)
    last = {L['firm']: L for L in res['log'][-2:]}
    rows.append(dict(first_mover=first, status=res['status'], iterations=res['iters'],
                     profit_R1=round(last['R1']['profit'], 1),
                     profit_R2=round(last['R2']['profit'], 1),
                     sales_R1=round(last['R1']['sales'], 1),
                     sales_R2=round(last['R2']['sales'], 1)))
pd.DataFrame(rows)

In [ ]:
res = cournot_iterate_miqp(first='R1', max_iter=16, env=ENV)
mo = pd.DataFrame(market_outcome(res['sales']))
print(f"total quantity {mo.quantity.sum():9.1f}   average price {mo.price.mean():6.2f}")
print(f"joint profit   {sum(L['profit'] for L in res['log'][-2:]):9.1f}")
mo.groupby('market')[['quantity', 'price', 'share_R1']].mean().round(3)

### Does the flooding result survive exact solution?

Part 4c's headline was that adding the production-learning channel raises output about 13% and
depresses price — firms sell past the static optimum because a unit sold advances them toward a
cheaper operating-cost tier. That was measured through the approximation. Re-measure it exactly.

In [ ]:
rows = []
for lm in ['capacity', 'both']:
    r2 = cournot_iterate_miqp(learning=lm, first='R1', max_iter=16, env=ENV)
    last = {L['firm']: L for L in r2['log'][-2:]}
    m2 = pd.DataFrame(market_outcome(r2['sales']))
    rows.append(dict(learning=lm, status=r2['status'],
                     total_quantity=round(m2.quantity.sum(), 1),
                     avg_price=round(m2.price.mean(), 2),
                     sales_R1=round(last['R1']['sales'], 1),
                     sales_R2=round(last['R2']['sales'], 1),
                     profit_R1=round(last['R1']['profit'], 1),
                     profit_R2=round(last['R2']['profit'], 1),
                     disposal=round(last['R1']['disposal']+last['R2']['disposal'], 2)))
pd.DataFrame(rows)

Compare these numbers against the piecewise results in Part 4c. What matters is not whether they
match to the digit — they will not — but whether the **qualitative** conclusions hold: output rising
with the production channel, price falling, the incumbent gaining more than the entrant, and
disposal remaining at zero.

A conclusion that survives both formulations is a conclusion about the economics. One that does not
was an artefact of the mesh.

## 6. Cournot against collusion, exactly

In [ ]:
jm = joint_profit_max()      # piecewise; see note below
res = cournot_iterate_miqp(first='R1', max_iter=16, env=ENV)
mo = pd.DataFrame(market_outcome(res['sales']))
jsales = {r: {(rt, p): jm._H[r]['sale'][rt, p].X for rt in REGIONS for p in P}
          for r in REGIONS}
mj = pd.DataFrame(market_outcome(jsales))
pd.DataFrame([
    dict(regime='Cournot (exact MIQP)', total_quantity=round(mo.quantity.sum(), 1),
         avg_price=round(mo.price.mean(), 2),
         joint_profit=round(sum(L['profit'] for L in res['log'][-2:]), 1)),
    dict(regime='Collusion (piecewise)', total_quantity=round(mj.quantity.sum(), 1),
         avg_price=round(mj.price.mean(), 2), joint_profit=round(jm.ObjVal, 1)),
])

One caveat on this table: the collusive benchmark is still solved with the piecewise revenue, so the
two rows are not computed identically. Given §4's finding that approximation error does not compose
predictably, the honest fix is to write an exact MIQP version of `joint_profit_max` as well —
straightforward, since its revenue is quadratic in the *sum* of the two firms' sales. Left as an
exercise rather than papered over.

## 7. Summary

| Question | Answer |
|---|---|
| Does the exact MIQP solve at full scale? | Yes, with a WLS licence — convex MIQP, native branch-and-bound |
| Does piecewise revenue understate profit? | **Yes, always** — concave and maximised, so chords lie below |
| How much does 7 breakpoints cost? | ~0.25% on a single best response |
| Does that error stay small in equilibrium? | **No.** It changes sign and grows to ~12% for R2 |
| Should Part 4c's conclusions be trusted? | The qualitative ones, yes — but verify each against this notebook |

### The transferable lesson

Approximation error inside an **optimisation** is signed and bounded. Approximation error inside an
**equilibrium computation** is neither, because each firm's error perturbs its rival's problem and
the perturbations compound around the best-response loop. Validate discretisations at the level you
are actually reporting.

That is also the argument for having the exact formulation available even when the approximation is
adequate: without it, there is no way to know that it *was* adequate.

### Things to try

- `NBP_REV = 3` in Part 4c and re-run its conclusions — how coarse before they break?
- Write `joint_profit_max_miqp` and complete §6 honestly
- `SMALL = True` — a 3-period instance where exact and approximate are both instant, useful for
  experimenting with the tolerance and cycle-detection settings
- With the WLS licence the whole series can scale: more scenarios in Part 2's extensive form, finer
  periods in Part 3, more tiers in Part 3b